In [2]:
!pip install deepface transformers git+https://github.com/openai/whisper.git
!sudo apt install ffmpeg

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-catbrad_
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-catbrad_
  Resolved https://github.com/openai/whisper.git to commit 517a43ecd132a2089d85f4ebc044728a71d49f6e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13

In [3]:
from IPython.display import display, Javascript
from google.colab.output import eval_js

def record_video(filename='video.webm'):
    js = Javascript("""
        async function recordVideo() {
            const div = document.createElement('div');
            const start = document.createElement('button');
            start.textContent = '🎥 Start Recording';
            div.appendChild(start);

            const stop = document.createElement('button');
            stop.textContent = '⏹️ Stop Recording';
            div.appendChild(stop);
            stop.style.display = 'none';

            const video = document.createElement('video');
            video.autoplay = true;
            video.style.width = '640px';
            div.appendChild(video);

            document.body.appendChild(div);

            const stream = await navigator.mediaDevices.getUserMedia({video: true, audio: true});
            video.srcObject = stream;

            await new Promise((resolve) => start.onclick = resolve);
            start.style.display = 'none';
            stop.style.display = 'inline';

            let recorder = new MediaRecorder(stream);
            let data = [];

            recorder.ondataavailable = event => data.push(event.data);
            recorder.start();

            await new Promise((resolve) => stop.onclick = resolve);
            recorder.stop();

            await new Promise(resolve => recorder.onstop = resolve);
            stream.getTracks().forEach(track => track.stop());
            div.remove();

            let blob = new Blob(data, {type: 'video/webm'});
            let arrayBuffer = await blob.arrayBuffer();
            return Array.from(new Uint8Array(arrayBuffer));
        }
    """)
    display(js)
    data = eval_js("recordVideo()")
    with open(filename, 'wb') as f:
        f.write(bytearray(data))
    print(f"Saved video to {filename}")


In [8]:
record_video("multi_input.webm")

<IPython.core.display.Javascript object>

Saved video to multi_input.webm


In [9]:
from IPython.display import HTML
from base64 import b64encode

def play_video(filename):
    mp4 = open(filename, 'rb').read()
    data_url = "data:video/webm;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <video width=640 controls>
        <source src="{data_url}" type="video/webm">
    </video>
    """)

play_video("multi_input.webm")


In [10]:
!ffmpeg -i multi_input.webm -qscale:v 3 -r 30 multi_input.mp4
!ffmpeg -i multi_input.webm -q:a 0 -map a audio.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [11]:
import cv2, os

def extract_frames(video_path="multi_input.mp4", out_folder="frames", frame_rate=1):
    os.makedirs(out_folder, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_interval = fps * frame_rate
    count = 0
    saved = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % frame_interval == 0:
            path = f"{out_folder}/frame_{saved}.jpg"
            cv2.imwrite(path, frame)
            saved += 1
        count += 1
    cap.release()
    print(f"{saved} frames saved to {out_folder}/")

extract_frames()


7 frames saved to frames/


In [12]:
from deepface import DeepFace
from collections import Counter
import os

def analyze_face_emotions(folder="frames"):
    emotions = []

    print("🔍 Analyzing facial expressions in video frames...\n")

    for f in sorted(os.listdir(folder)):
        if f.endswith(".jpg"):
            img_path = os.path.join(folder, f)
            try:
                result = DeepFace.analyze(img_path=img_path, actions=["emotion"], enforce_detection=False)
                dominant_emotion = result[0]["dominant_emotion"]
                emotions.append(dominant_emotion)
                print(f"{f}: {dominant_emotion}")
            except Exception as e:
                print(f"{f}: Skipped ({str(e)})")
                continue

    if emotions:
        most_common_emotion = Counter(emotions).most_common(1)[0][0]
        print(f"\n✅ Final Predicted Face-based Sentiment: {most_common_emotion}")
        return most_common_emotion
    else:
        print("\n⚠️ No detectable emotions found in frames.")
        return "unknown"

# Call the function
face_sentiment = analyze_face_emotions()


25-05-01 14:01:30 - Directory /root/.deepface has been created
25-05-01 14:01:30 - Directory /root/.deepface/weights has been created
🔍 Analyzing facial expressions in video frames...

25-05-01 14:01:31 - facial_expression_model_weights.h5 will be downloaded...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 39.5MB/s]


frame_0.jpg: neutral
frame_1.jpg: fear
frame_2.jpg: fear
frame_3.jpg: happy
frame_4.jpg: neutral
frame_5.jpg: happy
frame_6.jpg: happy

✅ Final Predicted Face-based Sentiment: happy


In [13]:
import whisper

model = whisper.load_model("base")
result = model.transcribe("audio.wav")
transcribed_text = result["text"]
print("Transcribed Speech:", transcribed_text)

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 78.4MiB/s]


Transcribed Speech:  I am very happy today.


In [14]:
from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis")

def get_text_sentiment(text):
    output = sentiment_model(text[:512])[0]  # BERT max input = 512 tokens
    return output['label'].lower()

text_sentiment = get_text_sentiment(transcribed_text)
print("Text-based Sentiment:", text_sentiment)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


Text-based Sentiment: positive


In [16]:
from collections import Counter

def final_sentiment_vote(face, text):
    # Give 7 votes to text sentiment and 3 votes to face sentiment
    votes = [text] * 7 + [face] * 3
    return Counter(votes).most_common(1)[0][0]

# Example usage
final_sentiment = final_sentiment_vote(face_sentiment, text_sentiment)
print("\n Final Predicted Sentiment (Multimodal):", final_sentiment)



 Final Predicted Sentiment (Multimodal): positive
